# The Prion Chameleon Lab: Modeling Alpha-to-Beta Transitions

Prions are among the most mysterious and dangerous entities in biology. Unlike viruses or bacteria, they contain no genetic material. Instead, they are misfolded versions of a normal cellular protein (**PrP<sup>C</sup>**) that can trigger a chain reaction, forcing healthy proteins to adopt the same toxic, misfolded shape (**PrP<sup>Sc</sup>**).

The hallmark of this transition is a drastic change in secondary structure: a protein that is primarily alpha-helical 'flips' into a dense, insoluble stack of beta-sheets (amyloid fibrils).

In this tutorial, we will use `synth-pdb` to model this 'Chameleon' behavior using a highly amyloidogenic fragment of the Human Prion Protein: **PrP 106-126**.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # @title Environment Setup
    !pip install -q synth-pdb biotite matplotlib numpy scipy py3Dmol openmm
else:
    sys.path.append(os.path.abspath("../../"))

print("✅ Environment configured!")

In [ ]:
import io
import py3Dmol
import numpy as np
import matplotlib.pyplot as plt
import biotite.structure.io.pdb as bpdb
from synth_pdb.generator import generate_pdb_content
from synth_pdb.chemical_shifts import predict_chemical_shifts
from synth_pdb.validator import PDBValidator

# The neurotoxic amyloidogenic core of human PrP
prion_sequence = "KTNMKHMAGAAAAGAVVGGLG"
print(f"Prion Fragment 106-126: {prion_sequence}")

## 1. Generating the Two Faces of a Prion

We will generate two distinct structural models for the *exact same sequence*:
1. **The Cellular Model**: An alpha-helical conformation.
2. **The Scrapie Model**: A beta-sheet conformation.

In [ ]:
# 1. Cellular (Alpha) state
alpha_pdb = generate_pdb_content(
    sequence_str=prion_sequence, conformation="alpha", minimize_energy=True
)

# 2. Scrapie (Beta) state
beta_pdb = generate_pdb_content(
    sequence_str=prion_sequence, conformation="beta", minimize_energy=True
)

print("Models generated and energy-minimized.")

## 2. Structural Comparison

Let's look at them side-by-side. The alpha-helix is a compact, self-satisfying coil, while the beta-strand is extended and 'sticky', ready to form hydrogen bonds with other strands to build a fibril.

In [ ]:
view = py3Dmol.view(width=800, height=400, linked=False, viewergrid=(1, 2))

# Left: Cellular (Alpha)
view.addModel(alpha_pdb, "pdb", viewer=(0, 0))
view.setStyle({"cartoon": {"color": "cyan"}}, viewer=(0, 0))
view.addLabel("PrPC (Alpha)", {"fontColor": "cyan", "backgroundColor": "black"}, viewer=(0, 0))
view.zoomTo(viewer=(0, 0))

# Right: Scrapie (Beta)
view.addModel(beta_pdb, "pdb", viewer=(0, 1))
view.setStyle({"cartoon": {"color": "magenta"}}, viewer=(0, 1))
view.addLabel("PrPSc (Beta)", {"fontColor": "magenta", "backgroundColor": "black"}, viewer=(0, 1))
view.zoomTo(viewer=(0, 1))

view.show()

## 3. The NMR Fingerprint

In the lab, NMR spectroscopy is the gold standard for detecting these transitions. We can calculate the **Chemical Shift Index (CSI)** for both models. Notice how the C-alpha atoms shift up-field in beta-sheets and down-field in alpha-helices.

In [ ]:
# predict_chemical_shifts expects a Biotite AtomArray, not a raw PDB string
struct_alpha = bpdb.PDBFile.read(io.StringIO(alpha_pdb)).get_structure(model=1)
struct_beta = bpdb.PDBFile.read(io.StringIO(beta_pdb)).get_structure(model=1)

shifts_alpha = predict_chemical_shifts(struct_alpha)
shifts_beta = predict_chemical_shifts(struct_beta)


# Extract CA shifts from the nested dict {chain: {res_id: {atom: value}}}
def extract_ca_shifts(shifts_dict):
    chain_shifts = next(iter(shifts_dict.values()), {})
    return [chain_shifts.get(r, {}).get("CA", 0.0) for r in sorted(chain_shifts)]


ca_alpha = extract_ca_shifts(shifts_alpha)
ca_beta = extract_ca_shifts(shifts_beta)
residues = np.arange(106, 106 + len(prion_sequence))

plt.figure(figsize=(10, 5))
plt.plot(residues, ca_alpha, "co-", label="PrPC (Alpha)")
plt.plot(residues, ca_beta, "mo-", label="PrPSc (Beta)")
plt.axhline(y=0, color="gray", linestyle="--")
plt.xlabel("Residue Number")
plt.ylabel("CA Chemical Shift Deviation (ppm)")
plt.title("NMR Signature of the Prion Transition")
plt.legend()
plt.grid(True)
plt.show()

print(
    "The large vertical gap between the lines represents the NMR detectable conformational change."
)

## 4. Modeling the 'Amyloid Nightmare'

Finally, let's generate a **Hard Decoy** that represents a catastrophic folding error—a sequence that *should* be helical but is forced into a sheet. This is exactly what PrP Scrapie does to healthy proteins.

In [ ]:
# Generate a decoy where the sequence is L-amino acids but forced into beta conformation
decoy_pdb = generate_pdb_content(
    sequence_str=prion_sequence,
    conformation="beta",
    drift=5.0,  # Add torsion noise to create structural variation
)

validator = PDBValidator(pdb_content=decoy_pdb)
report = validator.get_quality_report()
# report is a dict; "is_overall_scientifically_defensible" is the plausibility flag
plausible = report["is_overall_scientifically_defensible"]
rama_favored = report["ramachandran_stats"]["favored_pct"]
print(f"Decoy Plausible: {plausible}")
print(f"Ramachandran Favored: {rama_favored:.1f}%")
print(
    "Even though this is a toxic fold, it can be geometrically plausible, making it hard to detect!"
)